# CMNIST Results Analysis Notebook

Analyzing the results of checkpoints from IRO, GroupDRO, and ERM on CMNIST.

This notebook provides:
- recursive JSONL loading from `iro` artifact folders
- run health and seed coverage checks
- model-selection summaries at the chosen environment
- env-shift curves (`0.0..1.0`) for `best` and `final` checkpoints
- optional `iro eval` alpha sweep with baseline comparison
- CSV + figure export for reports


In [ ]:
# Setup: imports, plotting defaults, and display options
from __future__ import annotations

import json
import re
import subprocess
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn", "-q"])
    import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for cand in candidates:
        if (cand / "iro").exists() and (cand / "config").exists():
            return cand
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Notebook repo root: {REPO_ROOT}")


## Configuration

Update paths if needed. The loader scans recursively for `*.jsonl`.


In [ ]:
# Configuration
RESULTS_ROOT_CANDIDATES = [
    REPO_ROOT / "iro_exp" / "results",
    REPO_ROOT / "iro_exp_runs" / "results",
    REPO_ROOT / "iro_exp_test_cmnist" / "results",
    REPO_ROOT / "iro_exp_tmp2" / "results",
]

RESULTS_ROOT = next((p for p in RESULTS_ROOT_CANDIDATES if p.exists()), RESULTS_ROOT_CANDIDATES[0])
DATASET_FILTER = "cmnist"  # set to None to load all datasets
MODEL_SELECTION_ENV = 0.9
MODEL_SELECTION_ENV_LABEL = f"{MODEL_SELECTION_ENV:g}"
MODEL_SELECTION_STAGE = "best"  # "best" | "final"

OUT_DIR = REPO_ROOT / "collected_results" / "cmnist_analysis"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

ENV_GRID = [i / 10.0 for i in range(11)]

print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"RESULTS_ROOT exists: {RESULTS_ROOT.exists()}")
print(f"OUT_DIR: {OUT_DIR}")


## Loader Utilities

In [ ]:
ENV_METRIC_RE = re.compile(r"^(?P<env>-?(?:\d+(?:\.\d+)?|\.\d+))_(?P<metric>acc|loss)_(?P<stage>best|final|eval)$")
ALGORITHM_ORDER = [
    "erm",
    "groupdro",
    "iro",
    "inftask",
    "irm",
    "vrex",
    "arm",
    "coral",
    "eqrm",
    "iid",
    "mldg",
    "oracle",
]


def _is_scalar(v: Any) -> bool:
    return isinstance(v, (int, float, str, bool)) or v is None


def _to_float(v: Any) -> float | None:
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def _normalize_algorithm_label(raw_algorithm: Any, exp_name: Any) -> str:
    raw = str(raw_algorithm or "").strip().lower()
    exp = str(exp_name or "").strip().lower()

    for name in ALGORITHM_ORDER:
        if raw == name or raw.startswith(name + "(") or (name + "(") in raw:
            return "iro" if name == "inftask" else name

    for name in ALGORITHM_ORDER:
        if exp.endswith(f"_{name}") or f"_{name}_" in exp:
            return "iro" if name == "inftask" else name

    if "(" in raw:
        head = raw.split("(", 1)[0].strip()
        if head in ALGORITHM_ORDER:
            return "iro" if head == "inftask" else head

    return raw if raw else "unknown"


def iter_jsonl_paths(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(root.rglob("*.jsonl"))


def _parse_env_metric_key(key: str) -> tuple[str, str, str] | None:
    m = ENV_METRIC_RE.match(key)
    if m is None:
        return None
    env = m.group("env")
    metric = m.group("metric")
    stage = m.group("stage")
    return env, metric, stage


def _iter_history_metrics(eval_payload: Any):
    if isinstance(eval_payload, list):
        for metric in eval_payload:
            if isinstance(metric, dict):
                yield metric
    elif isinstance(eval_payload, dict):
        for env, metric in eval_payload.items():
            if not isinstance(metric, dict):
                continue
            payload = dict(metric)
            payload.setdefault("env", env)
            yield payload


def load_cmnist_records(results_root: Path, dataset_filter: str | None = "cmnist"):
    run_rows: list[dict[str, Any]] = []
    history_rows: list[dict[str, Any]] = []
    env_rows: list[dict[str, Any]] = []
    eval_rows: list[dict[str, Any]] = []
    error_rows: list[dict[str, Any]] = []

    for path in iter_jsonl_paths(results_root):
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    error_rows.append(
                        {
                            "file": str(path),
                            "line": line_no,
                            "error_type": "JSONDecodeError",
                            "error_message": "Could not parse JSON line",
                        }
                    )
                    continue

                result = rec.get("result")
                if not isinstance(result, dict):
                    result = {}

                dataset_name = rec.get("dataset_name") or result.get("dataset")
                if dataset_filter is not None and dataset_name != dataset_filter:
                    continue

                raw_algorithm = result.get("algorithm_name") or rec.get("algorithm")
                norm_algorithm = _normalize_algorithm_label(raw_algorithm, rec.get("exp_name"))

                run = {
                    "file": str(path),
                    "line": line_no,
                    "run_id": rec.get("run_id"),
                    "status": rec.get("status"),
                    "experiment": rec.get("experiment"),
                    "exp_name": rec.get("exp_name"),
                    "dataset_name": dataset_name,
                    "source": rec.get("source"),
                    "algorithm": norm_algorithm,
                    "algorithm_raw": str(raw_algorithm) if raw_algorithm is not None else None,
                    "seed": rec.get("seed"),
                    "args_id": rec.get("args_id"),
                    "output_root": rec.get("output_root"),
                    "selection_env": result.get("selection_env"),
                    "eval_split": result.get("split"),
                    "eval_alpha": _to_float(result.get("eval_alpha")),
                    "args": rec.get("args"),
                }

                for key, value in result.items():
                    if _is_scalar(value):
                        run[key] = value

                for source_name, payload in (("record", rec), ("result", result)):
                    if not isinstance(payload, dict):
                        continue
                    for key, value in payload.items():
                        if not isinstance(key, str):
                            continue
                        parsed = _parse_env_metric_key(key)
                        if parsed is None:
                            continue
                        env, metric, stage = parsed
                        numeric = _to_float(value)
                        if numeric is None:
                            continue
                        env_rows.append(
                            {
                                "run_id": run.get("run_id"),
                                "algorithm": run.get("algorithm"),
                                "seed": run.get("seed"),
                                "exp_name": run.get("exp_name"),
                                "stage": stage,
                                "metric": metric,
                                "env": env,
                                "value": numeric,
                                "source": source_name,
                            }
                        )
                        run.setdefault(key, numeric)

                run_rows.append(run)

                err = rec.get("error")
                if isinstance(err, dict):
                    error_rows.append(
                        {
                            "file": str(path),
                            "line": line_no,
                            "run_id": rec.get("run_id"),
                            "algorithm": run.get("algorithm"),
                            "seed": run.get("seed"),
                            "error_type": err.get("type"),
                            "error_message": err.get("message"),
                        }
                    )

                history = result.get("history")
                if isinstance(history, list):
                    for event in history:
                        if not isinstance(event, dict):
                            continue
                        train = event.get("train") if isinstance(event.get("train"), dict) else {}
                        selection_env = str(event.get("selection_env") or result.get("selection_env") or "")
                        for metric in _iter_history_metrics(event.get("eval")):
                            env = str(metric.get("env") or metric.get("split") or "")
                            history_rows.append(
                                {
                                    "run_id": run.get("run_id"),
                                    "algorithm": run.get("algorithm"),
                                    "seed": run.get("seed"),
                                    "exp_name": run.get("exp_name"),
                                    "step": _to_float(event.get("step")),
                                    "env": env,
                                    "selection_env": selection_env,
                                    "train_loss": _to_float(train.get("loss")),
                                    "acc": _to_float(metric.get("acc") if "acc" in metric else metric.get("accuracy")),
                                    "loss": _to_float(metric.get("loss")),
                                    "alpha": _to_float(metric.get("alpha")),
                                }
                            )

                for stage_key, stage in (("best_test_metrics", "best"), ("test_metrics", "final")):
                    stage_metrics = result.get(stage_key)
                    if not isinstance(stage_metrics, list):
                        continue
                    for metric in stage_metrics:
                        if not isinstance(metric, dict):
                            continue
                        env = str(metric.get("env") or "")
                        if not env:
                            continue
                        for metric_name in ("acc", "loss"):
                            numeric = _to_float(metric.get(metric_name))
                            if numeric is None:
                                continue
                            env_rows.append(
                                {
                                    "run_id": run.get("run_id"),
                                    "algorithm": run.get("algorithm"),
                                    "seed": run.get("seed"),
                                    "exp_name": run.get("exp_name"),
                                    "stage": stage,
                                    "metric": metric_name,
                                    "env": env,
                                    "value": numeric,
                                    "source": stage_key,
                                }
                            )

                eval_metrics = result.get("metrics")
                if isinstance(eval_metrics, list):
                    eval_alpha = _to_float(result.get("eval_alpha"))
                    eval_split = str(result.get("split") or "")
                    for metric in eval_metrics:
                        if not isinstance(metric, dict):
                            continue
                        env = str(metric.get("env") or metric.get("split") or "")
                        acc = _to_float(metric.get("acc") if "acc" in metric else metric.get("accuracy"))
                        loss = _to_float(metric.get("loss"))
                        alpha = eval_alpha if eval_alpha is not None else _to_float(metric.get("alpha"))
                        eval_rows.append(
                            {
                                "run_id": run.get("run_id"),
                                "algorithm": run.get("algorithm"),
                                "seed": run.get("seed"),
                                "exp_name": run.get("exp_name"),
                                "split": eval_split,
                                "env": env,
                                "acc": acc,
                                "loss": loss,
                                "eval_alpha": alpha,
                            }
                        )
                        if env and acc is not None:
                            env_rows.append(
                                {
                                    "run_id": run.get("run_id"),
                                    "algorithm": run.get("algorithm"),
                                    "seed": run.get("seed"),
                                    "exp_name": run.get("exp_name"),
                                    "stage": "eval",
                                    "metric": "acc",
                                    "env": env,
                                    "value": acc,
                                    "source": "eval_metrics",
                                }
                            )
                        if env and loss is not None:
                            env_rows.append(
                                {
                                    "run_id": run.get("run_id"),
                                    "algorithm": run.get("algorithm"),
                                    "seed": run.get("seed"),
                                    "exp_name": run.get("exp_name"),
                                    "stage": "eval",
                                    "metric": "loss",
                                    "env": env,
                                    "value": loss,
                                    "source": "eval_metrics",
                                }
                            )

    runs_df = pd.DataFrame(run_rows)
    history_df = pd.DataFrame(history_rows)
    env_df = pd.DataFrame(env_rows)
    eval_df = pd.DataFrame(eval_rows)
    errors_df = pd.DataFrame(error_rows)

    if not history_df.empty:
        history_df["step"] = pd.to_numeric(history_df["step"], errors="coerce")
        for col in ("train_loss", "acc", "loss", "alpha"):
            history_df[col] = pd.to_numeric(history_df[col], errors="coerce")

    if not env_df.empty:
        env_df["value"] = pd.to_numeric(env_df["value"], errors="coerce")
        env_df["env"] = env_df["env"].astype(str).str.strip()
        env_df = env_df.dropna(subset=["run_id", "stage", "metric", "env", "value"])
        env_df = (
            env_df.sort_values(["run_id", "stage", "metric", "env", "source"])
            .drop_duplicates(subset=["run_id", "stage", "metric", "env"], keep="first")
            .reset_index(drop=True)
        )

    if not eval_df.empty:
        for col in ("acc", "loss", "eval_alpha"):
            eval_df[col] = pd.to_numeric(eval_df[col], errors="coerce")

    return runs_df, history_df, env_df, eval_df, errors_df


In [ ]:
# Load all CMNIST records
runs_df, history_df, env_df, eval_df, errors_df = load_cmnist_records(RESULTS_ROOT, DATASET_FILTER)

print(f"runs rows: {len(runs_df):,}")
print(f"history rows: {len(history_df):,}")
print(f"env metric rows: {len(env_df):,}")
print(f"eval metric rows: {len(eval_df):,}")
print(f"error rows: {len(errors_df):,}")

if not runs_df.empty:
    display(runs_df.head(3))
if not env_df.empty:
    display(env_df.head(10))
if not errors_df.empty:
    print("Sample errors:")
    display(errors_df.head(5))


## Run Health and Coverage

In [ ]:
if runs_df.empty:
    print("No CMNIST records found under RESULTS_ROOT.")
else:
    health = (
        runs_df.groupby(["algorithm", "status"], dropna=False)
        .size()
        .rename("n")
        .reset_index()
        .sort_values(["algorithm", "status"])
    )
    display(health)

    coverage = (
        runs_df.pivot_table(index="algorithm", columns="seed", values="status", aggfunc="first")
        .sort_index()
    )
    display(coverage)


## Build Analysis View (Model-Selection Environment)

This merges flat columns (`0.9_acc_best`, `0.9_loss_best`, ...) with parsed env-level metrics.


In [ ]:
def build_selection_view(
    runs_df: pd.DataFrame,
    env_df: pd.DataFrame,
    *,
    env_label: str,
) -> pd.DataFrame:
    base_cols = [
        "run_id",
        "algorithm",
        "seed",
        "exp_name",
        "status",
        "output_root",
        "selection_env",
        "eval_alpha",
    ]
    existing_cols = [c for c in base_cols if c in runs_df.columns]
    base = runs_df[existing_cols].copy() if existing_cols else pd.DataFrame()
    if base.empty:
        return base

    for col in (
        "selection_acc_best",
        "selection_acc_final",
        "selection_loss_best",
        "selection_loss_final",
        "mean_env_acc_best",
        "mean_env_acc_final",
    ):
        base[col] = np.nan

    indexed = runs_df.set_index("run_id", drop=False) if "run_id" in runs_df.columns else pd.DataFrame()
    if not indexed.empty:
        for metric in ("acc", "loss"):
            for stage in ("best", "final"):
                source_col = f"{env_label}_{metric}_{stage}"
                target_col = f"selection_{metric}_{stage}"
                if source_col in indexed.columns:
                    mapped = pd.to_numeric(base["run_id"].map(indexed[source_col]), errors="coerce")
                    base[target_col] = base[target_col].where(base[target_col].notna(), mapped)

    if not env_df.empty:
        sel = env_df[(env_df["env"] == env_label) & (env_df["stage"].isin(["best", "final"]))].copy()
        if not sel.empty:
            sel_pivot = sel.pivot_table(index="run_id", columns=["metric", "stage"], values="value", aggfunc="mean")
            if not sel_pivot.empty:
                sel_pivot = sel_pivot.reset_index()
                sel_pivot.columns = ["run_id"] + [f"selection_{m}_{s}" for m, s in sel_pivot.columns.tolist()[1:]]
                base = base.merge(sel_pivot, on="run_id", how="left", suffixes=("", "_from_env"))
                for metric in ("acc", "loss"):
                    for stage in ("best", "final"):
                        col = f"selection_{metric}_{stage}"
                        alt = f"{col}_from_env"
                        if alt in base.columns:
                            base[col] = base[col].where(base[col].notna(), base[alt])
                            base = base.drop(columns=[alt])

        env_acc = env_df[(env_df["metric"] == "acc") & (env_df["stage"].isin(["best", "final"]))].copy()
        if not env_acc.empty:
            mean_pivot = (
                env_acc.groupby(["run_id", "stage"], as_index=False)["value"].mean()
                .pivot(index="run_id", columns="stage", values="value")
                .reset_index()
            )
            mean_pivot = mean_pivot.rename(columns={"best": "mean_env_acc_best", "final": "mean_env_acc_final"})
            base = base.merge(mean_pivot, on="run_id", how="left", suffixes=("", "_from_envmean"))
            for col in ("mean_env_acc_best", "mean_env_acc_final"):
                alt = f"{col}_from_envmean"
                if alt in base.columns:
                    base[col] = base[col].where(base[col].notna(), base[alt])
                    base = base.drop(columns=[alt])

    score_col = f"selection_acc_{MODEL_SELECTION_STAGE}"
    base["selection_score"] = pd.to_numeric(base.get(score_col), errors="coerce")
    return base


selection_view = build_selection_view(runs_df, env_df, env_label=MODEL_SELECTION_ENV_LABEL)
if "status" in selection_view.columns:
    selection_ok = selection_view[selection_view["status"] == "ok"].copy()
else:
    selection_ok = selection_view.copy()

selection_ok = selection_ok.sort_values(["algorithm", "seed", "selection_score"], ascending=[True, True, False])

print(f"selection_view rows: {len(selection_view):,}")
print(f"selection_ok rows: {len(selection_ok):,}")

if not selection_ok.empty:
    display(selection_ok.head(20))


## Algorithm Summary Table

In [ ]:
if selection_ok.empty:
    print("No successful runs available for summary.")
else:
    score_col = f"selection_acc_{MODEL_SELECTION_STAGE}"
    loss_col = f"selection_loss_{MODEL_SELECTION_STAGE}"

    summary = (
        selection_ok.groupby("algorithm", dropna=False)
        .agg(
            n_runs=("run_id", "count"),
            n_seeds=("seed", "nunique"),
            selection_acc_mean=(score_col, "mean"),
            selection_acc_std=(score_col, "std"),
            selection_loss_mean=(loss_col, "mean"),
            selection_loss_std=(loss_col, "std"),
            mean_env_acc_best=("mean_env_acc_best", "mean"),
            mean_env_acc_final=("mean_env_acc_final", "mean"),
        )
        .reset_index()
        .sort_values("selection_acc_mean", ascending=False)
    )
    display(summary)


## Visualization: Model-Selection Metrics by Algorithm

In [ ]:
if selection_ok.empty:
    print("No successful runs available for plotting.")
else:
    score_col = f"selection_acc_{MODEL_SELECTION_STAGE}"
    mean_col = f"mean_env_acc_{MODEL_SELECTION_STAGE}"

    plot_df = selection_ok.copy()
    order = (
        plot_df.groupby("algorithm")[score_col]
        .mean()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    metric_specs = [
        (score_col, f"Accuracy at env={MODEL_SELECTION_ENV_LABEL} ({MODEL_SELECTION_STAGE})", axes[0]),
        (mean_col, f"Mean accuracy over envs ({MODEL_SELECTION_STAGE})", axes[1]),
    ]

    for metric, title, ax in metric_specs:
        data = plot_df[["algorithm", "seed", metric]].dropna().copy()
        if data.empty:
            ax.set_title(f"{title} (no data)")
            continue

        stats = (
            data.groupby("algorithm", as_index=False)[metric]
            .agg(mean="mean", std="std")
            .set_index("algorithm")
            .reindex(order)
            .reset_index()
        )
        stats["std"] = stats["std"].fillna(0.0)

        palette = sns.color_palette("Set2", n_colors=len(stats))
        ax.bar(stats["algorithm"], stats["mean"], yerr=stats["std"], capsize=5, color=palette, alpha=0.9)

        sns.stripplot(
            data=data,
            x="algorithm",
            y=metric,
            order=order,
            color="black",
            alpha=0.45,
            size=4,
            jitter=0.15,
            ax=ax,
        )

        ax.set_title(title)
        ax.set_xlabel("algorithm")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=0.25)

    fig.savefig(FIG_DIR / "cmnist_selection_metrics_by_algorithm.png", dpi=150, bbox_inches="tight")
    plt.show()


## Visualization: Env-Shift Curves (`0.0..1.0`)

In [ ]:
if env_df.empty:
    print("No env-level metrics available for plotting.")
else:
    curves = env_df[(env_df["metric"] == "acc") & (env_df["stage"].isin(["best", "final"]))].copy()
    curves["env_float"] = pd.to_numeric(curves["env"], errors="coerce")
    curves = curves.dropna(subset=["env_float", "value"])

    if curves.empty:
        print("No usable env accuracy rows after filtering.")
    else:
        if "summary" in globals() and isinstance(summary, pd.DataFrame) and not summary.empty:
            alg_order = summary["algorithm"].tolist()
        else:
            alg_order = sorted(curves["algorithm"].dropna().unique().tolist())

        palette = sns.color_palette("tab10", n_colors=max(1, len(alg_order)))
        color_map = {alg: palette[i % len(palette)] for i, alg in enumerate(alg_order)}

        fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True, constrained_layout=True)

        for ax, stage in zip(axes, ["best", "final"]):
            stage_df = curves[curves["stage"] == stage].copy()
            if stage_df.empty:
                ax.set_title(f"{stage} (no data)")
                continue

            agg = (
                stage_df.groupby(["algorithm", "env_float"], as_index=False)["value"]
                .agg(mean="mean", std="std")
                .sort_values("env_float")
            )
            agg["std"] = agg["std"].fillna(0.0)

            for alg in alg_order:
                g = agg[agg["algorithm"] == alg]
                if g.empty:
                    continue
                c = color_map[alg]
                ax.plot(g["env_float"], g["mean"], marker="o", linewidth=2.0, color=c, label=alg)
                ax.fill_between(g["env_float"], g["mean"] - g["std"], g["mean"] + g["std"], alpha=0.18, color=c)

            ax.set_title(f"CMNIST env-shift accuracy ({stage})")
            ax.set_xlabel("environment value")
            ax.set_ylabel("accuracy")
            ax.set_ylim(0.0, 1.0)
            ax.grid(alpha=0.25)

        handles, labels = axes[0].get_legend_handles_labels()
        if labels:
            axes[0].legend(handles, labels, title="algorithm", fontsize=9)

        fig.savefig(FIG_DIR / "cmnist_env_shift_curves.png", dpi=150, bbox_inches="tight")
        plt.show()


## Training Dynamics from `result.history`

In [ ]:
if history_df.empty:
    print("No history rows found. This can happen if runs failed or used eval-only records.")
else:
    hist = history_df.copy()
    hist["env"] = hist["env"].astype(str)
    hist["selection_env"] = hist["selection_env"].fillna("").astype(str)

    selection_mask = (hist["selection_env"].str.len() > 0) & (hist["env"] == hist["selection_env"])
    sel_hist = hist[selection_mask].copy()

    if sel_hist.empty:
        sel_hist = hist[hist["env"] == MODEL_SELECTION_ENV_LABEL].copy()

    sel_hist = sel_hist.dropna(subset=["step"]).sort_values(["algorithm", "seed", "step"])

    if sel_hist.empty:
        print("No selection-environment history rows available.")
    else:
        agg = (
            sel_hist.groupby(["algorithm", "step"], as_index=False)
            .agg(
                acc_mean=("acc", "mean"),
                acc_std=("acc", "std"),
                loss_mean=("loss", "mean"),
                loss_std=("loss", "std"),
            )
            .sort_values("step")
        )
        agg[["acc_std", "loss_std"]] = agg[["acc_std", "loss_std"]].fillna(0.0)

        fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True, constrained_layout=True)
        algs = sorted(agg["algorithm"].dropna().unique().tolist())
        palette = sns.color_palette("tab10", n_colors=max(1, len(algs)))
        color_map = {alg: palette[i % len(palette)] for i, alg in enumerate(algs)}

        for alg in algs:
            g = agg[agg["algorithm"] == alg].sort_values("step")
            c = color_map[alg]
            axes[0].plot(g["step"], g["acc_mean"], linewidth=2.2, color=c, label=alg)
            axes[0].fill_between(g["step"], g["acc_mean"] - g["acc_std"], g["acc_mean"] + g["acc_std"], color=c, alpha=0.2)

            axes[1].plot(g["step"], g["loss_mean"], linewidth=2.2, color=c, label=alg)
            axes[1].fill_between(g["step"], g["loss_mean"] - g["loss_std"], g["loss_mean"] + g["loss_std"], color=c, alpha=0.2)

        axes[0].set_title(f"Selection-env accuracy vs step ({MODEL_SELECTION_ENV_LABEL})")
        axes[1].set_title(f"Selection-env loss vs step ({MODEL_SELECTION_ENV_LABEL})")
        axes[0].set_xlabel("step")
        axes[1].set_xlabel("step")
        axes[0].set_ylabel("accuracy")
        axes[1].set_ylabel("loss")
        axes[0].grid(alpha=0.25)
        axes[1].grid(alpha=0.25)
        axes[0].legend(title="algorithm", fontsize=9)

        fig.savefig(FIG_DIR / "cmnist_selection_env_training_dynamics.png", dpi=150, bbox_inches="tight")
        plt.show()


## IRO Alpha Sweep (eval mode) + Baseline Comparison

The `eval_df` table is empty until dedicated `iro eval` runs are created.

This section can generate those records directly from Python using `run_evaluation(...)`, cache them, and compare:
- IRO as a function of `eval.alpha`
- GroupDRO / ERM baselines from training runs


In [ ]:
# Run eval-mode alpha sweep for IRO checkpoints and cache results
try:
    from iro import run_evaluation
except ModuleNotFoundError:
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    from iro import run_evaluation

ALPHA_GRID = np.round(np.linspace(0.0, 1.0, 11), 2)
EVAL_SPLIT_FOR_SWEEP = "all"
ALPHA_TARGET_ENV_LABEL = MODEL_SELECTION_ENV_LABEL
RUN_IRO_ALPHA_SWEEP = False  # set True to launch eval runs from notebook
MAX_IRO_RUNS = None          # set integer to limit number of IRO checkpoints

EVAL_DATA_ROOT = REPO_ROOT / "data" / "cmnist"
EVAL_OUTPUT_ROOT = REPO_ROOT / "iro_exp_alpha_eval_cmnist"
EVAL_EXP_NAME = "cmnist_alpha_eval"
ALPHA_CACHE_CSV = OUT_DIR / "iro_alpha_eval_env_metrics.csv"


def checkpoint_path_from_row(row: pd.Series, which: str = "best") -> Path:
    output_root = Path(str(row.get("output_root") or (REPO_ROOT / "iro_exp")))
    return output_root / "ckpts" / f"{row['run_id']}_{which}.pkl"


iro_runs = selection_ok[(selection_ok["algorithm"] == "iro") & selection_ok["run_id"].notna()].copy()
if MAX_IRO_RUNS is not None and len(iro_runs):
    rank_col = f"selection_acc_{MODEL_SELECTION_STAGE}"
    if rank_col in iro_runs.columns:
        iro_runs = iro_runs.sort_values(rank_col, ascending=False).head(int(MAX_IRO_RUNS))

if len(iro_runs):
    iro_runs["ckpt_best"] = iro_runs.apply(checkpoint_path_from_row, axis=1)
    iro_runs["ckpt_exists"] = iro_runs["ckpt_best"].map(lambda p: p.exists())

display_cols = ["run_id", "seed", "exp_name", f"selection_acc_{MODEL_SELECTION_STAGE}", "ckpt_best", "ckpt_exists"]
if len(iro_runs):
    display(iro_runs[[c for c in display_cols if c in iro_runs.columns]])
else:
    print("No successful IRO runs found in selection view.")

alpha_eval_rows: list[dict[str, Any]] = []

if RUN_IRO_ALPHA_SWEEP:
    if iro_runs.empty:
        raise RuntimeError("No successful IRO runs found to evaluate.")

    for _, row in iro_runs.iterrows():
        ckpt = Path(row["ckpt_best"])
        if not ckpt.exists():
            print(f"Skipping missing checkpoint: {ckpt}")
            continue

        for alpha in ALPHA_GRID:
            overrides = [
                f"data.root={EVAL_DATA_ROOT}",
                "data.download=false",
                "iro.algorithm=iro",
                f"training.seed={int(row['seed']) if pd.notna(row['seed']) else 0}",
                "training.device=auto",
                f"training.output_root={EVAL_OUTPUT_ROOT}",
                f"training.exp_name={EVAL_EXP_NAME}",
                f"eval.checkpoint_path={ckpt}",
                f"eval.split={EVAL_SPLIT_FOR_SWEEP}",
                f"eval.alpha={float(alpha):.4f}",
            ]
            try:
                result = run_evaluation(experiment="cmnist_iro", overrides=overrides)
                metrics = result.get("metrics")
                if not isinstance(metrics, list):
                    metrics = []

                for metric in metrics:
                    if not isinstance(metric, dict):
                        continue
                    alpha_eval_rows.append(
                        {
                            "run_id": row["run_id"],
                            "seed": row.get("seed"),
                            "exp_name": row.get("exp_name"),
                            "eval_alpha": float(alpha),
                            "split": EVAL_SPLIT_FOR_SWEEP,
                            "env": str(metric.get("env") or ""),
                            "acc": _to_float(metric.get("acc") if "acc" in metric else metric.get("accuracy")),
                            "loss": _to_float(metric.get("loss")),
                            "checkpoint_path": str(ckpt),
                        }
                    )
            except Exception as exc:
                alpha_eval_rows.append(
                    {
                        "run_id": row["run_id"],
                        "seed": row.get("seed"),
                        "exp_name": row.get("exp_name"),
                        "eval_alpha": float(alpha),
                        "split": EVAL_SPLIT_FOR_SWEEP,
                        "env": ALPHA_TARGET_ENV_LABEL,
                        "acc": np.nan,
                        "loss": np.nan,
                        "checkpoint_path": str(ckpt),
                        "error": str(exc),
                    }
                )
                print(f"Eval failed for run_id={row['run_id']} alpha={alpha}: {exc}")

    alpha_eval_df = pd.DataFrame(alpha_eval_rows)
    if not alpha_eval_df.empty:
        alpha_eval_df.to_csv(ALPHA_CACHE_CSV, index=False)
        print(f"Saved alpha sweep cache: {ALPHA_CACHE_CSV}")
else:
    if ALPHA_CACHE_CSV.exists():
        alpha_eval_df = pd.read_csv(ALPHA_CACHE_CSV)
        print(f"Loaded cached alpha sweep: {ALPHA_CACHE_CSV}")
    else:
        alpha_eval_df = eval_df[eval_df["algorithm"] == "iro"].copy()
        if not alpha_eval_df.empty:
            print("Using existing eval_df rows for IRO alpha sweep (no fresh eval run).")
        else:
            alpha_eval_df = pd.DataFrame(
                columns=["run_id", "seed", "exp_name", "eval_alpha", "split", "env", "acc", "loss", "checkpoint_path"]
            )
            print("No eval-mode metrics detected yet. Set RUN_IRO_ALPHA_SWEEP=True to generate them.")

if not alpha_eval_df.empty:
    for col in ("eval_alpha", "acc", "loss"):
        alpha_eval_df[col] = pd.to_numeric(alpha_eval_df[col], errors="coerce")
    alpha_eval_df["env"] = alpha_eval_df["env"].astype(str)

display(alpha_eval_df.head(12))


In [ ]:
# Plot IRO alpha sweep against GroupDRO / ERM baselines
if alpha_eval_df.empty:
    print("No alpha-eval rows available. Run the previous cell with RUN_IRO_ALPHA_SWEEP=True.")
else:
    d = alpha_eval_df.dropna(subset=["eval_alpha", "acc"]).copy()
    d["env"] = d["env"].astype(str)

    target = d[d["env"] == ALPHA_TARGET_ENV_LABEL].copy()
    if target.empty:
        print(f"No alpha-eval rows found for target env={ALPHA_TARGET_ENV_LABEL}.")
    else:
        target_curve = (
            target.groupby("eval_alpha", as_index=False)["acc"]
            .agg(acc_mean="mean", acc_std="std")
            .sort_values("eval_alpha")
        )
        target_curve["acc_std"] = target_curve["acc_std"].fillna(0.0)

        overall = d.groupby(["run_id", "eval_alpha"], as_index=False)["acc"].mean()
        overall_curve = (
            overall.groupby("eval_alpha", as_index=False)["acc"]
            .agg(acc_mean="mean", acc_std="std")
            .sort_values("eval_alpha")
        )
        overall_curve["acc_std"] = overall_curve["acc_std"].fillna(0.0)

        score_col = f"selection_acc_{MODEL_SELECTION_STAGE}"
        mean_col = f"mean_env_acc_{MODEL_SELECTION_STAGE}"

        gd = selection_ok[selection_ok["algorithm"] == "groupdro"].copy()
        erm = selection_ok[selection_ok["algorithm"] == "erm"].copy()

        gd_target_mean = gd[score_col].mean() if score_col in gd.columns and not gd.empty else np.nan
        gd_target_std = gd[score_col].std(ddof=1) if score_col in gd.columns and len(gd) > 1 else 0.0
        erm_target_mean = erm[score_col].mean() if score_col in erm.columns and not erm.empty else np.nan

        gd_overall_mean = gd[mean_col].mean() if mean_col in gd.columns and not gd.empty else np.nan
        gd_overall_std = gd[mean_col].std(ddof=1) if mean_col in gd.columns and len(gd) > 1 else 0.0
        erm_overall_mean = erm[mean_col].mean() if mean_col in erm.columns and not erm.empty else np.nan

        x_min = float(target_curve["eval_alpha"].min())
        x_max = float(target_curve["eval_alpha"].max())

        fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

        axes[0].plot(
            target_curve["eval_alpha"],
            target_curve["acc_mean"],
            marker="o",
            linewidth=2.5,
            color="#1f77b4",
            label=f"IRO eval(alpha), env={ALPHA_TARGET_ENV_LABEL}",
        )
        axes[0].fill_between(
            target_curve["eval_alpha"],
            target_curve["acc_mean"] - target_curve["acc_std"],
            target_curve["acc_mean"] + target_curve["acc_std"],
            color="#1f77b4",
            alpha=0.2,
        )
        if np.isfinite(gd_target_mean):
            axes[0].hlines(gd_target_mean, x_min, x_max, colors="#d62728", linestyles="--", linewidth=2.0, label="GroupDRO")
            axes[0].fill_between([x_min, x_max], gd_target_mean - gd_target_std, gd_target_mean + gd_target_std, color="#d62728", alpha=0.14)
        if np.isfinite(erm_target_mean):
            axes[0].hlines(erm_target_mean, x_min, x_max, colors="#2ca02c", linestyles=":", linewidth=2.0, label="ERM")

        axes[0].set_title(f"Accuracy at env={ALPHA_TARGET_ENV_LABEL}")
        axes[0].set_xlabel("eval.alpha")
        axes[0].set_ylabel("accuracy")
        axes[0].grid(alpha=0.25)
        axes[0].legend()

        axes[1].plot(
            overall_curve["eval_alpha"],
            overall_curve["acc_mean"],
            marker="s",
            linewidth=2.5,
            color="#1f77b4",
            label="IRO eval(alpha), mean over envs",
        )
        axes[1].fill_between(
            overall_curve["eval_alpha"],
            overall_curve["acc_mean"] - overall_curve["acc_std"],
            overall_curve["acc_mean"] + overall_curve["acc_std"],
            color="#1f77b4",
            alpha=0.2,
        )
        if np.isfinite(gd_overall_mean):
            axes[1].hlines(gd_overall_mean, x_min, x_max, colors="#d62728", linestyles="--", linewidth=2.0, label="GroupDRO")
            axes[1].fill_between([x_min, x_max], gd_overall_mean - gd_overall_std, gd_overall_mean + gd_overall_std, color="#d62728", alpha=0.14)
        if np.isfinite(erm_overall_mean):
            axes[1].hlines(erm_overall_mean, x_min, x_max, colors="#2ca02c", linestyles=":", linewidth=2.0, label="ERM")

        axes[1].set_title("Mean accuracy over envs")
        axes[1].set_xlabel("eval.alpha")
        axes[1].set_ylabel("accuracy")
        axes[1].grid(alpha=0.25)
        axes[1].legend()

        fig.savefig(FIG_DIR / "iro_alpha_vs_baselines_cmnist.png", dpi=150, bbox_inches="tight")
        plt.show()


## DGIL-Style Risk Distribution on Env Losses

This approximates DGIL-style risk analysis from the alpha sweep by:
- taking eval losses for IRO across all envs
- summarizing per-alpha env-loss distributions
- plotting KDE-style curves and CVaR trend


In [ ]:
RISK_CACHE_CSV = OUT_DIR / "iro_env_risk_summary.csv"
RISK_FIG_PATH = FIG_DIR / "iro_env_risk_kde.png"
risk_summary_df = pd.DataFrame()


def _cvar(values: np.ndarray, alpha: float) -> float:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.nan
    q = np.quantile(values, alpha)
    tail = values[values >= q]
    return float(tail.mean()) if tail.size else float(q)


if "alpha_eval_df" in globals() and isinstance(alpha_eval_df, pd.DataFrame) and not alpha_eval_df.empty:
    risk_source = alpha_eval_df.dropna(subset=["eval_alpha", "env", "loss"]).copy()
    risk_source["env"] = risk_source["env"].astype(str)

    if risk_source.empty:
        print("Alpha sweep table has no loss values for risk analysis.")
    else:
        env_loss = (
            risk_source.groupby(["eval_alpha", "env"], as_index=False)["loss"]
            .mean()
            .sort_values(["eval_alpha", "env"])
        )

        alpha_to_risks: dict[float, np.ndarray] = {}
        summary_rows: list[dict[str, Any]] = []

        for alpha, grp in env_loss.groupby("eval_alpha"):
            values = grp["loss"].to_numpy(dtype=float)
            alpha_to_risks[float(alpha)] = values
            summary_rows.append(
                {
                    "alpha": float(alpha),
                    "n_envs": int(values.size),
                    "risk_mean": float(np.mean(values)) if values.size else np.nan,
                    "risk_std": float(np.std(values, ddof=1)) if values.size > 1 else 0.0,
                    "risk_cvar": _cvar(values, float(alpha)),
                }
            )

        risk_summary_df = pd.DataFrame(summary_rows).sort_values("alpha")
        risk_summary_df.to_csv(RISK_CACHE_CSV, index=False)
        display(risk_summary_df)

        fig, ax = plt.subplots(1, 1, figsize=(9, 6))
        ordered_alphas = sorted(alpha_to_risks.keys())
        colors = sns.color_palette("viridis", n_colors=max(1, len(ordered_alphas)))

        for color, alpha in zip(colors, ordered_alphas):
            values = alpha_to_risks.get(alpha, np.array([], dtype=float))
            if values.size == 0:
                continue
            if values.size > 2 and float(np.std(values)) > 1e-9:
                sns.kdeplot(values, ax=ax, color=color, linewidth=2.0)
            else:
                ax.axvline(float(np.mean(values)), color=color, linewidth=2.0)

        ax.set_title("Distribution of CMNIST Env Losses by Alpha")
        ax.set_xlabel("env loss")
        ax.set_ylabel("density")

        norm = plt.Normalize(float(np.min(ordered_alphas)), float(np.max(ordered_alphas)))
        sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        cbar.set_label(r"$\alpha$", labelpad=5, size=12)

        plt.tight_layout()
        fig.savefig(RISK_FIG_PATH, dpi=150, bbox_inches="tight")
        plt.show()

        plt.figure(figsize=(8, 5))
        plt.plot(risk_summary_df["alpha"], risk_summary_df["risk_cvar"], marker="o", linewidth=2.2)
        plt.title("CMNIST Env-Loss CVaR by Alpha")
        plt.xlabel("alpha")
        plt.ylabel("CVaR(env losses)")
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "iro_env_risk_cvar_vs_alpha.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    if RISK_CACHE_CSV.exists():
        risk_summary_df = pd.read_csv(RISK_CACHE_CSV)
        print(f"Loaded cached risk summary: {RISK_CACHE_CSV}")
        display(risk_summary_df)
    else:
        print("Risk summary cache not found. Run the alpha-sweep section first.")


## Export Tables

In [ ]:
runs_csv = OUT_DIR / "cmnist_runs_flat.csv"
history_csv = OUT_DIR / "cmnist_history_flat.csv"
env_csv = OUT_DIR / "cmnist_env_metrics_flat.csv"
eval_csv = OUT_DIR / "cmnist_eval_metrics_flat.csv"
errors_csv = OUT_DIR / "cmnist_errors_flat.csv"
selection_csv = OUT_DIR / "cmnist_selection_view.csv"

runs_df.to_csv(runs_csv, index=False)
history_df.to_csv(history_csv, index=False)
env_df.to_csv(env_csv, index=False)
eval_df.to_csv(eval_csv, index=False)
errors_df.to_csv(errors_csv, index=False)
selection_view.to_csv(selection_csv, index=False)

summary_csv = None
if "summary" in globals() and isinstance(summary, pd.DataFrame) and not summary.empty:
    summary_csv = OUT_DIR / "cmnist_algorithm_summary.csv"
    summary.to_csv(summary_csv, index=False)

alpha_eval_csv = None
if "alpha_eval_df" in globals() and isinstance(alpha_eval_df, pd.DataFrame) and not alpha_eval_df.empty:
    alpha_eval_csv = OUT_DIR / "iro_alpha_eval_env_metrics.csv"
    alpha_eval_df.to_csv(alpha_eval_csv, index=False)

risk_summary_csv = None
if "risk_summary_df" in globals() and isinstance(risk_summary_df, pd.DataFrame) and not risk_summary_df.empty:
    risk_summary_csv = OUT_DIR / "iro_env_risk_summary.csv"
    risk_summary_df.to_csv(risk_summary_csv, index=False)

print("Wrote:")
print(" -", runs_csv)
print(" -", history_csv)
print(" -", env_csv)
print(" -", eval_csv)
print(" -", errors_csv)
print(" -", selection_csv)
if summary_csv is not None:
    print(" -", summary_csv)
if alpha_eval_csv is not None:
    print(" -", alpha_eval_csv)
if risk_summary_csv is not None:
    print(" -", risk_summary_csv)
print("Figure directory:", FIG_DIR)


## Notes

- `MODEL_SELECTION_ENV` and `MODEL_SELECTION_STAGE` control summary/plot comparisons.
- Use `RUN_IRO_ALPHA_SWEEP=True` to generate fresh eval artifacts from checkpoints.
- Compare `best` and `final` env-shift curves before drawing conclusions about ranking stability.
